<a href="https://colab.research.google.com/github/Subuktageen-Farooqi/ms_course_deeplearning/blob/main/ms_deeplearning_tutorial_07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transfer Learning: Feature Extraction vs Fine Tuning - Tensorflow Implementation

In [ ]:
# Tutorial 7: Transfer Learning (TensorFlow / Keras)

import tensorflow as tf
import numpy as np
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.utils import to_categorical

# Step 1: Load CIFAR-10 dataset
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

# Normalize pixel values
train_images = train_images / 255.0
test_images = test_images / 255.0

# Create a validation split from the training set.
# Keep test split untouched for final evaluation only.
validation_split = 0.1
num_train = train_images.shape[0]
split_idx = int(num_train * (1 - validation_split))
rng = np.random.default_rng(42)
indices = rng.permutation(num_train)

train_idx = indices[:split_idx]
val_idx = indices[split_idx:]

val_images = train_images[val_idx]
val_labels = train_labels[val_idx]
train_images = train_images[train_idx]
train_labels = train_labels[train_idx]

# Convert labels to categorical
train_labels = to_categorical(train_labels, 10)
val_labels = to_categorical(val_labels, 10)
test_labels = to_categorical(test_labels, 10)


# Step 2: Feature Extraction using VGG16

# Load VGG16 pretrained model (without top layer)
vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Freeze convolutional base
vgg_base.trainable = False

# Build model
vgg_model = models.Sequential([
    vgg_base,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
])

# Compile
vgg_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train
vgg_model.fit(train_images, train_labels,
              epochs=10,
              validation_data=(val_images, val_labels))

# Evaluate
vgg_loss, vgg_acc = vgg_model.evaluate(test_images, test_labels)
print("VGG16 Feature Extraction Accuracy:", vgg_acc)


# Step 3: Fine-Tuning using ResNet50

# Load ResNet50 pretrained model (without top layer)
resnet_base = ResNet50(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Freeze all layers initially
for layer in resnet_base.layers:
    layer.trainable = False

# Build model
resnet_model = models.Sequential([
    resnet_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
])

# Compile
resnet_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train initial classifier
resnet_model.fit(train_images, train_labels,
                 epochs=10,
                 validation_data=(val_images, val_labels))


# Step 4: Fine-tuning (unfreeze some layers)

# Unfreeze last few layers
for layer in resnet_base.layers[-10:]:
    layer.trainable = True

# Recompile with lower learning rate
resnet_model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                     loss='categorical_crossentropy',
                     metrics=['accuracy'])

# Continue training
resnet_model.fit(train_images, train_labels,
                 epochs=10,
                 validation_data=(val_images, val_labels))

# Evaluate
resnet_loss, resnet_acc = resnet_model.evaluate(test_images, test_labels)
print("ResNet50 Fine-Tuning Accuracy:", resnet_acc)


# Step 5: Comparison

print("\nFinal Comparison:")
print(f"VGG16 Feature Extraction Accuracy: {vgg_acc:.4f}")
print(f"ResNet50 Fine-Tuning Accuracy: {resnet_acc:.4f}")

Epoch 1/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - accuracy: 0.5276 - loss: 1.3580 - val_accuracy: 0.5756 - val_loss: 1.2104
Epoch 2/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - accuracy: 0.5884 - loss: 1.1805 - val_accuracy: 0.5916 - val_loss: 1.1615
Epoch 3/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - accuracy: 0.6115 - loss: 1.1099 - val_accuracy: 0.6052 - val_loss: 1.1180
Epoch 4/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - accuracy: 0.6271 - loss: 1.0619 - val_accuracy: 0.6216 - val_loss: 1.0941
Epoch 5/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - accuracy: 0.6442 - loss: 1.0152 - val_accuracy: 0.6080 - val_loss: 1.1085
Epoch 6/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - accuracy: 0.6595 - loss: 0.9762 - val_accuracy: 0.6170 - val_loss: 1.0878
Epoch 7/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - accuracy: 0.6685 - loss: 0.9401 - val_accuracy: 0.6212 - val_loss: 1.0860
Epoch 8/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - accuracy: 0.6803 - loss: 

# Task 1: Transfer Learning PyTorch Implementation

In [1]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, models, transforms


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        #images, labels = images.to(device), labels.to(device)
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            #images, labels = images.to(device), labels.to(device)
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * labels.size(0)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total


def train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=5, tag="Model"):
    best_acc = 0.0
    best_weights = copy.deepcopy(model.state_dict())

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        if val_acc > best_acc:
            best_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())
        print(
            f"[{tag}] Epoch {epoch}/{epochs} | "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
        )

    model.load_state_dict(best_weights)
    return model, best_acc


def evaluate_test(model, test_loader, criterion, device, tag="Model"):
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    print(f"[{tag}] Final Test Loss: {test_loss:.4f}, Final Test Acc: {test_acc:.4f}")
    return test_acc


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    full_train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
    test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

    # take only a subset of the original train set, e.g. 10000 samples
    subset_size = 10000
    subset_indices = torch.randperm(len(full_train_dataset), generator=torch.Generator().manual_seed(42))[:subset_size]
    subset_dataset = Subset(full_train_dataset, subset_indices)

    val_size = int(0.1 * len(subset_dataset))
    train_size = len(subset_dataset) - val_size

    train_dataset, val_dataset = random_split(
        subset_dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42),
    )

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

    criterion = nn.CrossEntropyLoss()

    # Feature extraction with VGG16
    vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    for param in vgg.features.parameters():
        param.requires_grad = False

    in_features_vgg = vgg.classifier[-1].in_features
    vgg.classifier[-1] = nn.Linear(in_features_vgg, 10)
    vgg = vgg.to(device)

    vgg_params = [p for p in vgg.parameters() if p.requires_grad]
    vgg_optimizer = optim.Adam(vgg_params, lr=1e-3)
    vgg, vgg_best_acc = train_model(
        vgg, train_loader, val_loader, criterion, vgg_optimizer, device, epochs=5, tag="VGG16 Feature Extract"
    )
    vgg_test_acc = evaluate_test(vgg, test_loader, criterion, device, tag="VGG16 Feature Extract")

    # Fine-tuning with ResNet50: phase 1 (head only)
    resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    for param in resnet.parameters():
        param.requires_grad = False
    resnet.fc = nn.Linear(resnet.fc.in_features, 10)
    resnet = resnet.to(device)

    head_optimizer = optim.Adam(resnet.fc.parameters(), lr=1e-3)
    resnet, _ = train_model(
        resnet, train_loader, val_loader, criterion, head_optimizer, device, epochs=3, tag="ResNet50 Head Train"
    )

    # Fine-tuning with ResNet50: phase 2 (unfreeze last few layers)
    for name, param in resnet.named_parameters():
        if name.startswith("layer4") or name.startswith("fc"):
            param.requires_grad = True
        else:
            param.requires_grad = False

    ft_optimizer = optim.Adam((p for p in resnet.parameters() if p.requires_grad), lr=1e-5)
    resnet, resnet_best_acc = train_model(
        resnet, train_loader, val_loader, criterion, ft_optimizer, device, epochs=5, tag="ResNet50 Fine-Tune"
    )
    resnet_test_acc = evaluate_test(resnet, test_loader, criterion, device, tag="ResNet50 Fine-Tune")

    print("\n=== Final Comparison ===")
    print(f"VGG16 Best Validation Accuracy: {vgg_best_acc:.4f}")
    print(f"ResNet50 Best Validation Accuracy: {resnet_best_acc:.4f}")
    print(f"VGG16 Final Test Accuracy: {vgg_test_acc:.4f}")
    print(f"ResNet50 Final Test Accuracy: {resnet_test_acc:.4f}")
    if resnet_test_acc > vgg_test_acc:
        print("ResNet50 fine-tuning performed better on CIFAR-10 in this run.")
    elif resnet_test_acc < vgg_test_acc:
        print("VGG16 feature extraction performed better on CIFAR-10 in this run.")
    else:
        print("Both approaches achieved the same accuracy in this run.")


if __name__ == "__main__":
    main()

Using device: cuda


100%|██████████| 170M/170M [00:05<00:00, 29.0MB/s]


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 159MB/s]


[VGG16 Feature Extract] Epoch 1/5 | Train Loss: 0.8720, Train Acc: 0.7168 | Val Loss: 0.6786, Val Acc: 0.7780
[VGG16 Feature Extract] Epoch 2/5 | Train Loss: 0.4249, Train Acc: 0.8669 | Val Loss: 0.5691, Val Acc: 0.8180
[VGG16 Feature Extract] Epoch 3/5 | Train Loss: 0.3641, Train Acc: 0.9039 | Val Loss: 0.6229, Val Acc: 0.8550
[VGG16 Feature Extract] Epoch 4/5 | Train Loss: 0.3443, Train Acc: 0.9220 | Val Loss: 0.6953, Val Acc: 0.8440
[VGG16 Feature Extract] Epoch 5/5 | Train Loss: 0.2724, Train Acc: 0.9402 | Val Loss: 1.0501, Val Acc: 0.8190
[VGG16 Feature Extract] Final Test Loss: 0.6794, Final Test Acc: 0.8352
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 184MB/s]


[ResNet50 Head Train] Epoch 1/3 | Train Loss: 1.2883, Train Acc: 0.6366 | Val Loss: 0.9303, Val Acc: 0.7190
[ResNet50 Head Train] Epoch 2/3 | Train Loss: 0.8039, Train Acc: 0.7582 | Val Loss: 0.7709, Val Acc: 0.7400
[ResNet50 Head Train] Epoch 3/3 | Train Loss: 0.6913, Train Acc: 0.7807 | Val Loss: 0.7116, Val Acc: 0.7640
[ResNet50 Fine-Tune] Epoch 1/5 | Train Loss: 0.5765, Train Acc: 0.8166 | Val Loss: 0.6305, Val Acc: 0.7830
[ResNet50 Fine-Tune] Epoch 2/5 | Train Loss: 0.4772, Train Acc: 0.8482 | Val Loss: 0.5854, Val Acc: 0.8010
[ResNet50 Fine-Tune] Epoch 3/5 | Train Loss: 0.4033, Train Acc: 0.8744 | Val Loss: 0.5702, Val Acc: 0.8090
[ResNet50 Fine-Tune] Epoch 4/5 | Train Loss: 0.3389, Train Acc: 0.9001 | Val Loss: 0.5493, Val Acc: 0.8140
[ResNet50 Fine-Tune] Epoch 5/5 | Train Loss: 0.2879, Train Acc: 0.9134 | Val Loss: 0.5344, Val Acc: 0.8200
[ResNet50 Fine-Tune] Final Test Loss: 0.5314, Final Test Acc: 0.8177

=== Final Comparison ===
VGG16 Best Validation Accuracy: 0.8550
ResNet5

# Task 02: Improve Fine Tuning Performance

In [3]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

full_train_dataset = datasets.CIFAR10('./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10('./data', train=False, download=True, transform=transform)

# take only a subset of the original train set, e.g. 10000 samples
subset_size = 10000
subset_indices = torch.randperm(len(full_train_dataset), generator=torch.Generator().manual_seed(42))[:subset_size]
subset_dataset = Subset(full_train_dataset, subset_indices)

val_size = int(0.1 * len(subset_dataset))
train_size = len(subset_dataset) - val_size

train_dataset, val_dataset = random_split(
    subset_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

def run_epoch(model, loader, criterion, device, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(train_mode):
        for images, labels in loader:
            #images, labels = images.to(device), labels.to(device)
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            if train_mode:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train_mode:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

def train_with_best(model, train_loader, val_loader, criterion, optimizer, epochs=5, patience=None, tag='exp'):
    best_acc = 0.0
    best_state = copy.deepcopy(model.state_dict())
    wait = 0

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, device, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, device)
        print(f'[{tag}] Epoch {epoch}/{epochs} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}')

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if patience is not None and wait >= patience:
            print(f'[{tag}] Early stopping triggered at epoch {epoch}.')
            break

    model.load_state_dict(best_state)
    return model

def build_resnet50(num_classes=10):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


# Baseline: freeze all except FC, then unfreeze layer4+fc with standard settings
baseline = build_resnet50().to(device)
for p in baseline.parameters():
    p.requires_grad = False
for p in baseline.fc.parameters():
    p.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer_head = optim.Adam((p for p in baseline.parameters() if p.requires_grad), lr=1e-3)
_ = train_with_best(baseline, train_loader, val_loader, criterion, optimizer_head, epochs=3, tag='baseline-head')

for name, p in baseline.named_parameters():
    p.requires_grad = name.startswith('layer4') or name.startswith('fc')

optimizer_ft = optim.Adam((p for p in baseline.parameters() if p.requires_grad), lr=1e-4)
baseline = train_with_best(baseline, train_loader, val_loader, criterion, optimizer_ft, epochs=5, tag='baseline-ft')
_, baseline_acc = run_epoch(baseline, test_loader, criterion, device)
print('Baseline test accuracy:', baseline_acc)


Device: cuda
[baseline-head] Epoch 1/3 | Train Acc: 0.6222 | Val Acc: 0.7260
[baseline-head] Epoch 2/3 | Train Acc: 0.7553 | Val Acc: 0.7500
[baseline-head] Epoch 3/3 | Train Acc: 0.7840 | Val Acc: 0.7620
[baseline-ft] Epoch 1/5 | Train Acc: 0.8330 | Val Acc: 0.8460
[baseline-ft] Epoch 2/5 | Train Acc: 0.9670 | Val Acc: 0.8570
[baseline-ft] Epoch 3/5 | Train Acc: 0.9930 | Val Acc: 0.8510
[baseline-ft] Epoch 4/5 | Train Acc: 0.9980 | Val Acc: 0.8500
[baseline-ft] Epoch 5/5 | Train Acc: 0.9978 | Val Acc: 0.8660
Baseline test accuracy: 0.8506


In [4]:
"""
Experiment Combination 1
Methods used: lower learning rate + early stopping.
"""
combo1 = build_resnet50().to(device)
for p in combo1.parameters():
    p.requires_grad = False

for name, p in combo1.named_parameters():
    if name.startswith('layer4') or name.startswith('fc'):
        p.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer_combo1 = optim.Adam((p for p in combo1.parameters() if p.requires_grad), lr=5e-5)
combo1 = train_with_best(
    combo1, train_loader, val_loader, criterion, optimizer_combo1, epochs=12, patience=3, tag='combo1'
)
_, combo1_acc = run_epoch(combo1, test_loader, criterion, device)
print('Combination 1 test accuracy:', combo1_acc)


[combo1] Epoch 1/12 | Train Acc: 0.6112 | Val Acc: 0.7920
[combo1] Epoch 2/12 | Train Acc: 0.8452 | Val Acc: 0.8430
[combo1] Epoch 3/12 | Train Acc: 0.9191 | Val Acc: 0.8500
[combo1] Epoch 4/12 | Train Acc: 0.9636 | Val Acc: 0.8570
[combo1] Epoch 5/12 | Train Acc: 0.9826 | Val Acc: 0.8630
[combo1] Epoch 6/12 | Train Acc: 0.9919 | Val Acc: 0.8590
[combo1] Epoch 7/12 | Train Acc: 0.9957 | Val Acc: 0.8660
[combo1] Epoch 8/12 | Train Acc: 0.9961 | Val Acc: 0.8640
[combo1] Epoch 9/12 | Train Acc: 0.9969 | Val Acc: 0.8630
[combo1] Epoch 10/12 | Train Acc: 0.9979 | Val Acc: 0.8600
[combo1] Early stopping triggered at epoch 10.
Combination 1 test accuracy: 0.8532


In [6]:
"""
Experiment Combination 2
Methods used: unfreeze more layers + overfitting prevention (dropout + weight decay)
"""

combo2 = build_resnet50().to(device)
# Add dropout in the classification head for regularization
combo2.fc = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(combo2.fc.in_features, 10)
)
combo2 = combo2.to(device)

for p in combo2.parameters():
    p.requires_grad = False

# Unfreeze more layers than baseline: layer3, layer4, and fc
for name, p in combo2.named_parameters():
    if name.startswith('layer3') or name.startswith('layer4') or name.startswith('fc'):
        p.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer_combo2 = optim.Adam((p for p in combo2.parameters() if p.requires_grad), lr=1e-4, weight_decay=1e-4)
combo2 = train_with_best(combo2, train_loader, val_loader, criterion, optimizer_combo2, epochs=10, tag='combo2')
_, combo2_acc = run_epoch(combo2, test_loader, criterion, device)
print('Combination 2 test accuracy:', combo2_acc)

[combo2] Epoch 1/10 | Train Acc: 0.6908 | Val Acc: 0.8920
[combo2] Epoch 2/10 | Train Acc: 0.9361 | Val Acc: 0.8920
[combo2] Epoch 3/10 | Train Acc: 0.9846 | Val Acc: 0.8920
[combo2] Epoch 4/10 | Train Acc: 0.9907 | Val Acc: 0.9000
[combo2] Epoch 5/10 | Train Acc: 0.9937 | Val Acc: 0.9040
[combo2] Epoch 6/10 | Train Acc: 0.9911 | Val Acc: 0.9000
[combo2] Epoch 7/10 | Train Acc: 0.9910 | Val Acc: 0.8980
[combo2] Epoch 8/10 | Train Acc: 0.9942 | Val Acc: 0.8940
[combo2] Epoch 9/10 | Train Acc: 0.9941 | Val Acc: 0.9030
[combo2] Epoch 10/10 | Train Acc: 0.9951 | Val Acc: 0.9020
Combination 2 test accuracy: 0.9035


# Task 03: Implement feature extraction with Resnet50 & compare the results with custom model and VGG16

In [7]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models, transforms


class CustomCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


def run_epoch(model, loader, criterion, device, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            #images, labels = images.to(device), labels.to(device)
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            if is_train:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total


def train_and_eval(model, train_loader, val_loader, test_loader, epochs, lr, device, tag, weight_decay=0.0):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam((p for p in model.parameters() if p.requires_grad), lr=lr, weight_decay=weight_decay)
    model = model.to(device)

    best_acc = 0.0
    best_state = copy.deepcopy(model.state_dict())
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, device, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, device)
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
        print(
            f"[{tag}] Epoch {epoch}/{epochs} | "
            f"Train Loss: {tr_loss:.4f}, Train Acc: {tr_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
        )

    model.load_state_dict(best_state)
    _, test_acc = run_epoch(model, test_loader, criterion, device)
    return test_acc


def build_vgg16_feature_extractor():
    model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    for p in model.features.parameters():
        p.requires_grad = False
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 10)
    return model


def build_resnet50_feature_extractor():
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, 10)
    return model


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    tf_custom = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])
    tf_pretrained = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    train_custom = datasets.CIFAR10("./data", train=True, download=True, transform=tf_custom)
    test_custom = datasets.CIFAR10("./data", train=False, download=True, transform=tf_custom)
    train_pre = datasets.CIFAR10("./data", train=True, download=True, transform=tf_pretrained)
    test_pre = datasets.CIFAR10("./data", train=False, download=True, transform=tf_pretrained)

    custom_train_len = int(0.9 * len(train_custom))
    custom_val_len = len(train_custom) - custom_train_len
    pre_train_len = int(0.9 * len(train_pre))
    pre_val_len = len(train_pre) - pre_train_len

    custom_train_ds, custom_val_ds = random_split(
        train_custom, [custom_train_len, custom_val_len], generator=torch.Generator().manual_seed(42)
    )
    pre_train_ds, pre_val_ds = random_split(
        train_pre, [pre_train_len, pre_val_len], generator=torch.Generator().manual_seed(42)
    )

    train_custom_loader = DataLoader(custom_train_ds, batch_size=128, shuffle=True, num_workers=2)
    val_custom_loader = DataLoader(custom_val_ds, batch_size=256, shuffle=False, num_workers=2)
    test_custom_loader = DataLoader(test_custom, batch_size=256, shuffle=False, num_workers=2)

    train_pre_loader = DataLoader(pre_train_ds, batch_size=64, shuffle=True, num_workers=2)
    val_pre_loader = DataLoader(pre_val_ds, batch_size=128, shuffle=False, num_workers=2)
    test_pre_loader = DataLoader(test_pre, batch_size=128, shuffle=False, num_workers=2)

    custom_model = CustomCNN(num_classes=10)
    custom_acc = train_and_eval(custom_model, train_custom_loader, val_custom_loader, test_custom_loader, epochs=8, lr=1e-3, device=device, tag="Custom CNN")

    vgg_model = build_vgg16_feature_extractor()
    vgg_acc = train_and_eval(vgg_model, train_pre_loader, val_pre_loader, test_pre_loader, epochs=5, lr=1e-3, device=device, tag="VGG16 Feature Extract")

    resnet_model = build_resnet50_feature_extractor()
    resnet_acc = train_and_eval(resnet_model, train_pre_loader, val_pre_loader, test_pre_loader, epochs=5, lr=1e-3, device=device, tag="ResNet50 Feature Extract")

    print("\n=== Task 3 Final Comparison (CIFAR-10 Test Accuracy) ===")
    print("{:<35} {:>10}".format("Model", "Accuracy"))
    print("-" * 48)
    print("{:<35} {:>10.4f}".format("Custom CNN (from scratch)", custom_acc))
    print("{:<35} {:>10.4f}".format("VGG16 Feature Extraction", vgg_acc))
    print("{:<35} {:>10.4f}".format("ResNet50 Feature Extraction", resnet_acc))


if __name__ == "__main__":
    main()

Using device: cuda
[Custom CNN] Epoch 1/8 | Train Loss: 1.8270, Train Acc: 0.3148 | Val Loss: 1.6464, Val Acc: 0.3826
[Custom CNN] Epoch 2/8 | Train Loss: 1.5524, Train Acc: 0.4303 | Val Loss: 1.4936, Val Acc: 0.4508
[Custom CNN] Epoch 3/8 | Train Loss: 1.4304, Train Acc: 0.4798 | Val Loss: 1.4129, Val Acc: 0.4914
[Custom CNN] Epoch 4/8 | Train Loss: 1.3446, Train Acc: 0.5132 | Val Loss: 1.3345, Val Acc: 0.5206
[Custom CNN] Epoch 5/8 | Train Loss: 1.2630, Train Acc: 0.5470 | Val Loss: 1.2430, Val Acc: 0.5612
[Custom CNN] Epoch 6/8 | Train Loss: 1.2067, Train Acc: 0.5721 | Val Loss: 1.2030, Val Acc: 0.5734
[Custom CNN] Epoch 7/8 | Train Loss: 1.1540, Train Acc: 0.5916 | Val Loss: 1.1513, Val Acc: 0.5928
[Custom CNN] Epoch 8/8 | Train Loss: 1.1227, Train Acc: 0.6017 | Val Loss: 1.1308, Val Acc: 0.6024
[VGG16 Feature Extract] Epoch 1/5 | Train Loss: 0.7217, Train Acc: 0.7716 | Val Loss: 0.4543, Val Acc: 0.8506
[VGG16 Feature Extract] Epoch 2/5 | Train Loss: 0.5301, Train Acc: 0.8464 | Val

# Task 04: Implement Fine Tuning with VGG16

In [9]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models, transforms


def epoch_pass(model, loader, criterion, device, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for images, labels in loader:
            #images, labels = images.to(device), labels.to(device)
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            if training:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total


def train_model(model, train_loader, val_loader, optimizer, criterion, device, epochs, tag):
    best_acc = 0.0
    best_state = copy.deepcopy(model.state_dict())

    for ep in range(1, epochs + 1):
        tr_loss, tr_acc = epoch_pass(model, train_loader, criterion, device, optimizer)
        val_loss, val_acc = epoch_pass(model, val_loader, criterion, device)
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"[{tag}] Epoch {ep}/{epochs} | "
            f"Train Loss: {tr_loss:.4f}, Train Acc: {tr_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
        )

    model.load_state_dict(best_state)
    return model


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    full_train_dataset = datasets.CIFAR10("./data", train=True, download=True, transform=transform)
    test_dataset = datasets.CIFAR10("./data", train=False, download=True, transform=transform)

    # take only a subset of the original train set, e.g. 10000 samples
    subset_size = 10000
    subset_indices = torch.randperm(len(full_train_dataset), generator=torch.Generator().manual_seed(42))[:subset_size]
    subset_dataset = Subset(full_train_dataset, subset_indices)

    val_size = int(0.1 * len(subset_dataset))
    train_size = len(subset_dataset) - val_size

    train_dataset, val_dataset = random_split(
        subset_dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42),
    )

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

    criterion = nn.CrossEntropyLoss()

    # VGG16 fine-tuning (selected later layers + classifier)
    vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    vgg.classifier[-1] = nn.Linear(vgg.classifier[-1].in_features, 10)

    for p in vgg.features.parameters():
        p.requires_grad = False
    for p in vgg.features[24:].parameters():  # unfreeze later conv blocks for fine-tuning
        p.requires_grad = True
    for p in vgg.classifier.parameters():
        p.requires_grad = True

    vgg = vgg.to(device)
    vgg_optimizer = optim.Adam((p for p in vgg.parameters() if p.requires_grad), lr=1e-4, weight_decay=1e-4)
    vgg = train_model(vgg, train_loader, val_loader, vgg_optimizer, criterion, device, epochs=6, tag="VGG16 Fine-Tune")
    _, vgg_acc = epoch_pass(vgg, test_loader, criterion, device)

    # ResNet50 feature extraction (head only)
    resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    for p in resnet.parameters():
        p.requires_grad = False
    resnet.fc = nn.Linear(resnet.fc.in_features, 10)
    resnet = resnet.to(device)

    resnet_optimizer = optim.Adam(resnet.fc.parameters(), lr=1e-3)
    resnet = train_model(
        resnet, train_loader, val_loader, resnet_optimizer, criterion, device, epochs=5, tag="ResNet50 Feature Extract"
    )
    _, resnet_acc = epoch_pass(resnet, test_loader, criterion, device)

    print("\n=== Task 4 Comparison Summary ===")
    print(f"VGG16 Fine-Tuning Test Accuracy: {vgg_acc:.4f}")
    print(f"ResNet50 Feature Extraction Test Accuracy: {resnet_acc:.4f}")


if __name__ == "__main__":
    main()

Using device: cuda
[VGG16 Fine-Tune] Epoch 1/6 | Train Loss: 0.7132, Train Acc: 0.7523 | Val Loss: 0.4091, Val Acc: 0.8610
[VGG16 Fine-Tune] Epoch 2/6 | Train Loss: 0.2523, Train Acc: 0.9146 | Val Loss: 0.4113, Val Acc: 0.8560
[VGG16 Fine-Tune] Epoch 3/6 | Train Loss: 0.0810, Train Acc: 0.9736 | Val Loss: 0.4231, Val Acc: 0.8820
[VGG16 Fine-Tune] Epoch 4/6 | Train Loss: 0.0438, Train Acc: 0.9844 | Val Loss: 0.5769, Val Acc: 0.8620
[VGG16 Fine-Tune] Epoch 5/6 | Train Loss: 0.0311, Train Acc: 0.9893 | Val Loss: 0.6255, Val Acc: 0.8390
[VGG16 Fine-Tune] Epoch 6/6 | Train Loss: 0.0272, Train Acc: 0.9920 | Val Loss: 0.5343, Val Acc: 0.8740
[ResNet50 Feature Extract] Epoch 1/5 | Train Loss: 1.2868, Train Acc: 0.6360 | Val Loss: 0.9126, Val Acc: 0.7350
[ResNet50 Feature Extract] Epoch 2/5 | Train Loss: 0.8118, Train Acc: 0.7543 | Val Loss: 0.7730, Val Acc: 0.7520
[ResNet50 Feature Extract] Epoch 3/5 | Train Loss: 0.6883, Train Acc: 0.7882 | Val Loss: 0.7167, Val Acc: 0.7660
[ResNet50 Feature 